# 01 — Workspace setup (offline-first)

This notebook demonstrates the expected runtime layout for `meridian_expert` while staying fully offline:

- synthetic fixture repositories (`meridian`, `meridian_aux`),
- deterministic fake LLM backend,
- environment-variable and local-config path concepts,
- a `doctor` health check against the synthetic workspace.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import tempfile
from pathlib import Path
from textwrap import dedent


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not find repository root from current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
WORK_ROOT = Path(tempfile.mkdtemp(prefix='meridian_expert_demo_'))
WORKSPACE = WORK_ROOT / 'runtime'

os.environ['PYTHONPATH'] = str(REPO_ROOT / 'src')
os.environ['MERIDIAN_EXPERT_WORKSPACE'] = str(WORKSPACE)
os.environ['MERIDIAN_EXPERT_LLM_BACKEND'] = 'fake'
os.environ.setdefault('OPENAI_API_KEY', 'demo-not-used-with-fake-backend')

from meridian_expert.testing_support.repo_fixtures import build_fixture_workspace

repos = build_fixture_workspace(WORK_ROOT)
os.environ['MERIDIAN_REPO_PATH'] = str(repos['meridian'])
os.environ['MERIDIAN_AUX_REPO_PATH'] = str(repos['meridian_aux'])


def run_cli(args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    cmd = ['python', '-m', 'meridian_expert.cli', *shlex.split(args)]
    result = subprocess.run(cmd, cwd=REPO_ROOT, env=os.environ.copy(), text=True, capture_output=True)
    print('$', ' '.join(cmd))
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print('--- stderr ---')
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}: {args}')
    return result


def write_demo_task(relative_path: str, body: str) -> Path:
    path = WORK_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dedent(body).strip() + '
', encoding='utf-8')
    return path


def read_text(path: Path, max_chars: int = 1600) -> None:
    text = path.read_text(encoding='utf-8')
    print(text[:max_chars] + ('
... (truncated)' if len(text) > max_chars else ''))


print('Repo root:', REPO_ROOT)
print('Demo workspace:', WORK_ROOT)
print('Meridian fixture:', repos['meridian'])
print('Meridian_aux fixture:', repos['meridian_aux'])


## Expected runtime layout and environment variables

In [ ]:
print('MERIDIAN_EXPERT_WORKSPACE=', os.environ['MERIDIAN_EXPERT_WORKSPACE'])
print('MERIDIAN_REPO_PATH=', os.environ['MERIDIAN_REPO_PATH'])
print('MERIDIAN_AUX_REPO_PATH=', os.environ['MERIDIAN_AUX_REPO_PATH'])
print('MERIDIAN_EXPERT_LLM_BACKEND=', os.environ['MERIDIAN_EXPERT_LLM_BACKEND'])


## Local config concept (`config/repos.local.yaml`)

In [ ]:
sample = dedent(f'''
meridian_repo_path: {repos['meridian']}
meridian_aux_repo_path: {repos['meridian_aux']}
workspace_path: {WORKSPACE}
''').strip()
print(sample)


## Run doctor

In [ ]:
run_cli('doctor')